In [2]:
import sys, re, hashlib
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from src.embeddings import encoder_textes, encoder_images, decouper_texte

PDF_PATH  = ROOT / "data" / "attention_is_all_you_need.pdf"
TEXT_DIR  = ROOT / "resultats_pdf" / "textes"
IMAGE_DIR = ROOT / "resultats_pdf" / "images"

In [4]:
import chromadb

client = chromadb.PersistentClient(path=str(ROOT / "chroma_db"))

collection = client.get_or_create_collection(
    name="pdf_clip_vit_b32",
    embedding_function=None,            # c'est NOUS qui calculons les vecteurs
    metadata={"hnsw:space": "cosine"},  # distance utilisée pour la recherche
)
print("Éléments déjà présents :", collection.count())

Éléments déjà présents : 0


In [12]:
document_id = hashlib.sha256(PDF_PATH.read_bytes()).hexdigest()[:24]
print("document_id :", document_id)

# On supprime l'ancienne indexation de CE document avant de réindexer
collection.delete(where={"document_id": document_id})

document_id : bdfaa68d8984f0dc02beaca5


{'deleted': 166}

In [13]:
def numero_page(chemin):
    m = re.search(r"page_(\d+)", Path(chemin).stem)
    return int(m.group(1)) if m else -1

ids, documents, metadatas = [], [], []

for fichier in sorted(TEXT_DIR.glob("page_*.txt")):
    page = numero_page(fichier)
    for i, chunk in enumerate(decouper_texte(fichier.read_text(encoding="utf-8"))):
        ids.append(f"{document_id}:texte:{page}:{i}")
        documents.append(chunk)
        metadatas.append({
            "type": "texte",
            "document_id": document_id,
            "pdf": PDF_PATH.name,
            "page": page,
            "fragment": i,
        })

TAILLE_LOT = 64
for d in range(0, len(documents), TAILLE_LOT):
    lot = documents[d:d + TAILLE_LOT]
    collection.upsert(
        ids=ids[d:d + TAILLE_LOT],
        documents=lot,
        metadatas=metadatas[d:d + TAILLE_LOT],
        embeddings=encoder_textes(lot).tolist(),
    )

print(len(documents), "chunks de texte indexés")

163 chunks de texte indexés


In [14]:
EXTENSIONS = {".png", ".jpg", ".jpeg", ".webp", ".bmp"}
images = [f for f in sorted(IMAGE_DIR.iterdir()) if f.suffix.lower() in EXTENSIONS]

collection.upsert(
    ids=[f"{document_id}:image:{f.name}" for f in images],
    embeddings=encoder_images(images).tolist(),
    documents=[f"Figure extraite du PDF, page {numero_page(f)}" for f in images],
    metadatas=[{
        "type": "image",
        "document_id": document_id,
        "pdf": PDF_PATH.name,
        "page": numero_page(f),
        "chemin": f.relative_to(ROOT).as_posix(),
    } for f in images],
)
print(len(images), "images indexées | total :", collection.count())

3 images indexées | total : 166


In [15]:
print("Total :", collection.count())

imgs = collection.get(where={"type": "image"}, include=["metadatas"])
for m in imgs["metadatas"]:
    print(m)

collection.peek(2)["documents"]

Total : 166
{'chemin': 'resultats_pdf/images/page_003_image_128.png', 'page': 3, 'document_id': 'bdfaa68d8984f0dc02beaca5', 'pdf': 'attention_is_all_you_need.pdf', 'type': 'image'}
{'type': 'image', 'document_id': 'bdfaa68d8984f0dc02beaca5', 'page': 4, 'chemin': 'resultats_pdf/images/page_004_image_182.png', 'pdf': 'attention_is_all_you_need.pdf'}
{'pdf': 'attention_is_all_you_need.pdf', 'type': 'image', 'document_id': 'bdfaa68d8984f0dc02beaca5', 'page': 4, 'chemin': 'resultats_pdf/images/page_004_image_183.png'}


['provided proper attribution is provided , google hereby grants permission to reproduce the tables and figures in this paper solely for use in journalistic or scholarly works . attention is all you need ashish vaswani ∗ noam shazeer ∗ niki parmar ∗ jakob uszkoreit ∗ 2 0 2 3 google brain google brain google research google research aug avas',
 'brain google brain google research google research aug avaswani @ google . comllion jones ∗ noam @ google . comaidan n . gomez ∗† nikip @ google . comłukaszusz @ google . comkaiser ∗ 2 google research university of toronto google brain llion @ google . com aidan @ cs . toronto . edu lu']